In [70]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json
import os

class TicTacToeDataset(Dataset):
    def __init__(self, filepath):
        self.inputs, self.targets = [], []
        with open(filepath, 'r') as f:
            games = json.load(f)
            for game in games:
                for i in range(1, len(game)):
                    self.inputs.append(game[:i])
                    self.targets.append(game[i])
                    
    def __len__(self): return len(self.inputs)
    
    def __getitem__(self, idx):
        seq = self.inputs[idx]
        padded_seq = seq + [10] * (10 - len(seq)) 
        return torch.tensor(padded_seq, dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

# In a notebook, paths are relative to the notebook file's location
filepath = '../data/games.json'
dataset = TicTacToeDataset(filepath)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"Loaded {len(dataset)} training examples.")

Loaded 76379 training examples.


In [71]:
class TinyTicTacToeGPT(nn.Module):
    # We added d_model, num_layers, and nhead as arguments here
    def __init__(self, d_model=64, num_layers=2, nhead=4):
        super().__init__()
        
        # 1. Update Embeddings to use d_model
        self.embedding = nn.Embedding(11, d_model) 
        self.pos_encoder = nn.Embedding(10, d_model)
        
        # 2. Update Transformer Layer to use d_model and nhead
        decoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        
        # 3. Update Transformer Encoder to use num_layers
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 4. Update the final output layer to take d_model as input
        self.fc_out = nn.Linear(d_model, 11)
        
    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoder(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        out = self.transformer(x, mask=mask)
        return self.fc_out(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [76]:
model = TinyTicTacToeGPT(d_model=128, num_layers=3, nhead=8).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=10) 
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0002)

print("Starting training...")
for epoch in range(15): 
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        logits = model(inputs)
        lengths = (inputs != 10).sum(dim=1)
        batch_indices = torch.arange(inputs.size(0))
        last_step_logits = logits[batch_indices, lengths - 1]
        
        loss = criterion(last_step_logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/15 - Loss: {total_loss/len(dataloader):.4f}")
print("Training complete!")

Starting training...
Epoch 1/15 - Loss: 1.6304
Epoch 2/15 - Loss: 1.4216
Epoch 3/15 - Loss: 1.3767
Epoch 4/15 - Loss: 1.3669
Epoch 5/15 - Loss: 1.3621
Epoch 6/15 - Loss: 1.3560
Epoch 7/15 - Loss: 1.3553
Epoch 8/15 - Loss: 1.3512
Epoch 9/15 - Loss: 1.3482
Epoch 10/15 - Loss: 1.3485
Epoch 11/15 - Loss: 1.3468
Epoch 12/15 - Loss: 1.3433
Epoch 13/15 - Loss: 1.3436
Epoch 14/15 - Loss: 1.3410
Epoch 15/15 - Loss: 1.3405
Training complete!


In [78]:
def probe_model(sequence):
    model.eval()
    with torch.no_grad():
        padded_seq = sequence + [10] * (10 - len(sequence))
        inputs = torch.tensor(padded_seq).unsqueeze(0).to(device)
        logits = model(inputs)
        
        step_logits = logits[0, len(sequence) - 1]
        probs = torch.softmax(step_logits, dim=0)
        
        print(f"\n--- Testing Game State: {sequence} ---")
        for i in range(10):
            if i == 9:
                status = "ILLEGAL (Already ended)" if 9 in sequence else "Legal (End Game)"
                name = "End Token (9)"
            else:
                status = "ILLEGAL (Already played)" if i in sequence else "Legal"
                name = f"Square {i}    "
                
            print(f"{name} ({status}): {probs[i].item()*100:.1f}%")

# Change these to whatever you want and re-run this specific cell!
probe_model([4, 0, 8])
probe_model([8, 6, 1, 4, 7, 3, 0, 2])


--- Testing Game State: [4, 0, 8] ---
Square 0     (ILLEGAL (Already played)): 0.1%
Square 1     (Legal): 16.1%
Square 2     (Legal): 22.2%
Square 3     (Legal): 17.1%
Square 4     (ILLEGAL (Already played)): 0.1%
Square 5     (Legal): 10.8%
Square 6     (Legal): 19.4%
Square 7     (Legal): 14.2%
Square 8     (ILLEGAL (Already played)): 0.1%
End Token (9) (Legal (End Game)): 0.0%

--- Testing Game State: [8, 6, 1, 4, 7, 3, 0, 2] ---
Square 0     (ILLEGAL (Already played)): 0.0%
Square 1     (ILLEGAL (Already played)): 0.0%
Square 2     (ILLEGAL (Already played)): 0.0%
Square 3     (ILLEGAL (Already played)): 0.0%
Square 4     (ILLEGAL (Already played)): 0.0%
Square 5     (Legal): 0.0%
Square 6     (ILLEGAL (Already played)): 0.0%
Square 7     (ILLEGAL (Already played)): 0.0%
Square 8     (ILLEGAL (Already played)): 0.0%
End Token (9) (Legal (End Game)): 99.9%


In [79]:
torch.save(model.state_dict(), '../models/transformer/tictactoe_model.pth')
print("Model saved successfully as 'tictactoe_model.pth'!")

Model saved successfully as 'tictactoe_model.pth'!


In [80]:
probe_model([0, 3, 1, 4, 2])
probe_model([0, 1, 4, 2, 8])
probe_model([0, 3, 1, 2, 4, 8, 7])
probe_model([0, 1, 4, 8, 3, 5, 6])


--- Testing Game State: [0, 3, 1, 4, 2] ---
Square 0     (ILLEGAL (Already played)): 0.0%
Square 1     (ILLEGAL (Already played)): 0.0%
Square 2     (ILLEGAL (Already played)): 0.0%
Square 3     (ILLEGAL (Already played)): 0.0%
Square 4     (ILLEGAL (Already played)): 0.0%
Square 5     (Legal): 0.0%
Square 6     (Legal): 0.0%
Square 7     (Legal): 0.0%
Square 8     (Legal): 0.0%
End Token (9) (Legal (End Game)): 99.9%

--- Testing Game State: [0, 1, 4, 2, 8] ---
Square 0     (ILLEGAL (Already played)): 0.0%
Square 1     (ILLEGAL (Already played)): 0.0%
Square 2     (ILLEGAL (Already played)): 0.0%
Square 3     (Legal): 0.0%
Square 4     (ILLEGAL (Already played)): 0.0%
Square 5     (Legal): 0.0%
Square 6     (Legal): 0.0%
Square 7     (Legal): 0.0%
Square 8     (ILLEGAL (Already played)): 0.0%
End Token (9) (Legal (End Game)): 99.9%

--- Testing Game State: [0, 3, 1, 2, 4, 8, 7] ---
Square 0     (ILLEGAL (Already played)): 0.0%
Square 1     (ILLEGAL (Already played)): 0.0%
Square 2   